# Download Qwen3-4B without a GPU

This notebook downloads the post-trained `Qwen/Qwen3-4B` checkpoint from ModelScope to a local Transformers-compatible directory. It never loads the model, so it can run on a CPU-only instance.

The default destination is `Qwen3-4B/` under the current working directory and matches `MODEL_SOURCE` in `client_math.ipynb`. Re-running the download reuses completed files and resumes missing content.


In [ ]:
# Papermill parameters
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = "master"
DESTINATION = "Qwen3-4B"
INSTALL_MODELSCOPE_IF_MISSING = True


## Prepare the CPU-only downloader


In [ ]:
import os

# The downloader does not need CUDA. Hiding GPUs also prevents accidental model loading.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("modelscope") is None:
    if not INSTALL_MODELSCOPE_IF_MISSING:
        raise ImportError("modelscope is not installed")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "modelscope"])

destination = Path(DESTINATION).expanduser().resolve()
destination.mkdir(parents=True, exist_ok=True)
print(f"Python:      {sys.executable}")
print(f"Repository:  {MODEL_ID}")
print(f"Revision:    {MODEL_REVISION}")
print(f"Destination: {destination}")


## Download or resume

ModelScope writes directly to the destination. Existing complete files are reused; interrupted downloads can be resumed by running this cell again.


In [ ]:
from modelscope import snapshot_download

snapshot_path = snapshot_download(
    model_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=str(destination),
)
print(f"\nSnapshot ready at: {snapshot_path}", flush=True)


## Confirm the downloaded weight shards


In [ ]:
import json

required_metadata = [
    destination / "config.json",
    destination / "tokenizer_config.json",
]
missing_metadata = [str(path) for path in required_metadata if not path.is_file()]
if missing_metadata:
    raise FileNotFoundError("Missing model metadata: " + ", ".join(missing_metadata))

index_path = destination / "model.safetensors.index.json"
single_weight = destination / "model.safetensors"
if index_path.is_file():
    index = json.loads(index_path.read_text(encoding="utf-8"))
    shard_names = sorted(set(index.get("weight_map", {}).values()))
    shard_paths = [destination / name for name in shard_names]
elif single_weight.is_file():
    shard_paths = [single_weight]
else:
    raise FileNotFoundError("No safetensors weight index or model.safetensors was found")

missing_shards = [str(path) for path in shard_paths if not path.is_file() or path.stat().st_size == 0]
if missing_shards:
    raise FileNotFoundError("Missing or empty weight shards: " + ", ".join(missing_shards))

total_bytes = sum(path.stat().st_size for path in shard_paths)
print(f"Verified weight files: {len(shard_paths)}")
for path in shard_paths:
    print(f"  {path.name}: {path.stat().st_size / 2**30:.3f} GiB")
print(f"Total weight size: {total_bytes / 2**30:.3f} GiB")
print("\nUse this in client_math.ipynb:")
print(f'MODEL_SOURCE = "{destination}"')
